In [1]:
%pip install ultralytics

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path

# Замените путь на папку, где лежат изображения и XML
SOURCE_DIRS = [
    Path(r"C:\Users\user\Desktop\detects_ns\archive\dataset_2\main_dataset_2\synthetic_defect"),
]

OUTPUT_DIR = Path("yolo_dataset")

In [3]:
import xml.etree.ElementTree as ET
import random
import shutil

image_extensions = [".png", ".jpg", ".jpeg", ".bmp"]
pairs = []

for folder in SOURCE_DIRS:
    if not folder.exists():
        raise FileNotFoundError(f"Папка не найдена: {folder}")

    for xml_path in folder.glob("*.xml"):
        image_path = next(
            (folder / f"{xml_path.stem}{ext}" for ext in image_extensions
             if (folder / f"{xml_path.stem}{ext}").exists()),
            None
        )
        if image_path:
            pairs.append((image_path, xml_path))

if not pairs:
    raise RuntimeError("Не найдены пары изображений и XML. Проверьте SOURCE_DIRS.")

# Создать структуру папок заново
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

for split in ("train", "val"):
    (OUTPUT_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

random.seed(42)
random.shuffle(pairs)

if len(pairs) == 1:
    train_pairs, val_pairs = pairs, pairs
else:
    split_index = max(1, min(len(pairs) - 1, int(len(pairs) * 0.8)))
    train_pairs, val_pairs = pairs[:split_index], pairs[split_index:]

def convert_xml_to_yolo(xml_path, label_path):
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    image_width = float(size.findtext("width"))
    image_height = float(size.findtext("height"))

    lines = []

    for obj in root.findall("object"):
        if obj.findtext("name") != "defect":
            continue

        box = obj.find("bndbox")
        xmin = float(box.findtext("xmin"))
        ymin = float(box.findtext("ymin"))
        xmax = float(box.findtext("xmax"))
        ymax = float(box.findtext("ymax"))

        x_center = ((xmin + xmax) / 2) / image_width
        y_center = ((ymin + ymax) / 2) / image_height
        box_width = (xmax - xmin) / image_width
        box_height = (ymax - ymin) / image_height

        lines.append(
            f"0 {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}"
        )

    label_path.write_text("\n".join(lines), encoding="utf-8")

def copy_pairs(items, split):
    for image_path, xml_path in items:
        shutil.copy2(image_path, OUTPUT_DIR / "images" / split / image_path.name)
        convert_xml_to_yolo(
            xml_path,
            OUTPUT_DIR / "labels" / split / f"{image_path.stem}.txt"
        )

copy_pairs(train_pairs, "train")
copy_pairs(val_pairs, "val")

data_yaml = OUTPUT_DIR / "data.yaml"
data_yaml.write_text(
    f"path: {OUTPUT_DIR.resolve().as_posix()}\n"
    "train: images/train\n"
    "val: images/val\n"
    "names:\n"
    "  0: defect\n",
    encoding="utf-8"
)

print(f"Всего изображений: {len(pairs)}")
print(f"Обучение: {len(train_pairs)}")
print(f"Проверка: {len(val_pairs)}")
print(f"Конфигурация: {data_yaml.resolve()}")

Всего изображений: 129
Обучение: 103
Проверка: 26
Конфигурация: C:\Users\user\Desktop\detects_ns\no_simple_version\yolo_dataset\data.yaml


In [4]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # небольшая модель, подходит для первого запуска

results = model.train(
    data=str(OUTPUT_DIR / "data.yaml"),
    epochs=100,
    imgsz=640,
    batch=8,
    name="defect_detector"
)

Ultralytics 8.4.187  Python-3.11.0 torch-2.4.1+cpu CPU (11th Gen Intel Core i5-11400 @ 2.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset\data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=defect_detector, nbs=64, nms=None, opset=None, optimize

In [6]:
BEST_MODEL = Path("runs/detect/defect_detector/weights/best.pt")

# Укажите путь к фото, на котором нужно найти дефект
TEST_IMAGE = Path(r"C:\Users\user\Desktop\detects_ns\no_simple_version\test.jpg")

model = YOLO(str(BEST_MODEL))
predictions = model.predict(
    source=str(TEST_IMAGE),
    conf=0.25,
    save=True
)

print("Результаты сохранены в папке runs/detect/predict")


image 1/1 C:\Users\user\Desktop\detects_ns\no_simple_version\test.jpg: 512x640 2 defects, 52.1ms
Speed: 1.9ms preprocess, 52.1ms inference, 0.9ms postprocess per image at shape (1, 3, 512, 640)
Results saved to C:\Users\user\Desktop\detects_ns\no_simple_version\runs\detect\predict-2
Результаты сохранены в папке runs/detect/predict
